In [ ]:
# ============================================================
# CELL 1 — SETUP + LOAD TEST QUESTIONS
# Creates Mistral/zero-shot folder and loads fixed test set
# KAGGLE VERSION
# ============================================================

!pip install -q "transformers>=4.48,<5" accelerate bitsandbytes \
    huggingface_hub sacrebleu rapidfuzz bert-score==0.3.13

import os, re, gc, json, unicodedata
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# ---------- Paths ----------
# Kaggle datasets are mounted under /kaggle/input.
# This automatically finds test_questions.csv anywhere under /kaggle/input.
INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/Mistral/zero-shot")
OUT.mkdir(parents=True, exist_ok=True)

matches = list(INPUT_ROOT.rglob("test_questions.csv"))

if not matches:
    raise FileNotFoundError(
        "Could not find test_questions.csv under /kaggle/input. "
        "Add the dataset containing test_questions.csv to this Kaggle notebook."
    )

if len(matches) > 1:
    print("Multiple test_questions.csv files found:")
    for p in matches:
        print(" -", p)
    print("Using:", matches[0])

TEST_PATH = matches[0]

# ---------- Load fixed test set ----------
tests = pd.read_csv(TEST_PATH).fillna("")

assert "question" in tests.columns
assert "gold" in tests.columns
assert (tests["gold"].astype(str).str.strip() != "").all()

print("Test file:", TEST_PATH)
print("Test questions:", len(tests))
print("Output folder:", OUT)

display(tests.head())


In [ ]:
# ============================================================
# CELL 2 — MISTRAL-7B-INSTRUCT-V0.3 ZERO-SHOT INFERENCE
# No examples, no RAG, no fine-tuning
# max_new_tokens = 500
# Saves checkpoint + predictions.csv
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from tqdm.auto import tqdm

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_NEW_TOKENS = 500

if not torch.cuda.is_available():
    raise RuntimeError("Please enable GPU accelerator in Kaggle.")

print("GPU:", torch.cuda.get_device_name(0))

# ---------- 4-bit loading ----------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading Mistral-7B-Instruct-v0.3...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

device = model.get_input_embeddings().weight.device


# ---------- Zero-shot prompt ----------
def generate_answer(question):

    messages = [
        {
            "role": "system",
            "content":
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক একজন সহকারী। "
                "প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
                "প্রয়োজন হলে সঠিক ফি, সময়, প্রয়োজনীয় কাগজপত্র "
                "এবং প্রক্রিয়া উল্লেখ করো। "
                "অপ্রয়োজনীয় ব্যাখ্যা দিও না।"
        },
        {
            "role": "user",
            "content": f"প্রশ্ন: {question}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and generated[-1].item() != tokenizer.eos_token_id
    )

    return answer, truncated


# ============================================================
# RESUME SUPPORT
# ============================================================

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(PARTIAL).fillna("")

    for _, row in old.iterrows():
        done[
            (str(row["domain"]), str(row["id"]))
        ] = row.to_dict()

print("Already completed:", len(done))


# ============================================================
# GENERATE
# ============================================================

predictions = []

for _, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Mistral Zero-Shot"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result

    predictions.append(result)

    # checkpoint after every question
    pd.DataFrame(predictions).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ---------- Final predictions ----------
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCompleted:", len(pred_df))

print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print("Saved:", OUT / "predictions.csv")


# Free GPU before evaluation
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL 3 — EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1,
# BERT Precision, Recall and F1
# Saves result.csv
# ============================================================

from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (df["gold"].astype(str).str.strip() != "").all()


# ---------- Normalization ----------
BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(BN_TO_EN).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(r"\s+", " ", text).strip()


def tokens(text):
    return normalize(text).split()


# ---------- Exact Match ----------
def exact_match(pred, gold):
    return float(
        normalize(pred) == normalize(gold)
    )


# ---------- Token F1 ----------
def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-N ----------
def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum((pn & gn).values())

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------- ROUGE-L ----------
def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)
            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# ROW-LEVEL METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(df["prediction"], df["gold"])
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ============================================================
# BERTSCORE
# ============================================================

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ============================================================
# FINAL RESULT TABLE
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


# Save row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall metrics
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(OUT / "predictions.csv")
print(OUT / "result.csv")
